# BorrowSmart State-Level Prototype Pipeline

State-level prototype pipeline for a fictional nonprofit, BorrowSmart, using public FSA, ACS, and CFPB datasets to generate advisor-facing debt burden summaries.

**v2 — reviewed by K. Baisden.** Changes from v1:
- Added error handling around data loading (try/except with descriptive messages)
- Documented risk label threshold rationale
- Chart now exports to PNG file in addition to inline display
- Minor code comments added for clarity

## Prototype Success Metrics

- successful ingestion of all 3 source files in one run
- retention of all 50 states after joins (excluding District of Columbia if present in source data)

- zero missing values in critical exported fields used for the debt burden indicator
- automatic generation of a full state-level output table and top 10 burden-state table
- pipeline runtime below a defined threshold during prototype execution


## Section 1 - Setup

Upload the three source files to Colab:
- `portfolio-by-location.xls`
- `ACSDT5Y2024.B19013-Data.csv`
- `map_data_STU.csv`


In [29]:
%pip -q install xlrd


In [30]:
import io
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pipeline_start = time.time()

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)


In [31]:
# Colab upload (with local fallback for non-Colab execution)
try:
    from google.colab import files  # type: ignore
    print('Upload the 3 required files now...')
    uploaded = files.upload()
    uploaded_names = set(uploaded.keys())
except ImportError:
    uploaded = {}
    uploaded_names = set()
    print('Not running in Colab. Using local fallback paths if available.')

REQUIRED_FILES = {
    'fsa': 'portfolio-by-location.xls',
    'acs': 'ACSDT5Y2024.B19013-Data.csv',
    'cfpb': 'map_data_STU.csv',
}

LOCAL_FALLBACK_DIR = Path('data/raw')


def resolve_input_path(filename: str) -> Path:
    if filename in uploaded_names:
        return Path(filename)
    if Path(filename).exists():
        return Path(filename)
    fallback = LOCAL_FALLBACK_DIR / filename
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f'Missing required file: {filename}. Upload it in Colab or place it in {LOCAL_FALLBACK_DIR}.')


FSA_FILE = resolve_input_path(REQUIRED_FILES['fsa'])
ACS_FILE = resolve_input_path(REQUIRED_FILES['acs'])
CFPB_FILE = resolve_input_path(REQUIRED_FILES['cfpb'])

print('Resolved files:')
print('  FSA :', FSA_FILE)
print('  ACS :', ACS_FILE)
print('  CFPB:', CFPB_FILE)

pipeline_start = time.time()


Upload the 3 required files now...


Saving ACSDT5Y2024.B19013-Data.csv to ACSDT5Y2024.B19013-Data (2).csv
Saving map_data_STU.csv to map_data_STU (2).csv
Saving portfolio-by-location.xls to portfolio-by-location (2).xls
Resolved files:
  FSA : portfolio-by-location.xls
  ACS : ACSDT5Y2024.B19013-Data.csv
  CFPB: map_data_STU.csv


## Section 2 - Load Data

Load each dataset, clean headers/non-data rows, and extract the required fields.


In [ ]:
STATE_NAME_TO_FIPS = {
    'Alabama': 1, 'Alaska': 2, 'Arizona': 4, 'Arkansas': 5, 'California': 6, 'Colorado': 8,
    'Connecticut': 9, 'Delaware': 10, 'District of Columbia': 11, 'Florida': 12, 'Georgia': 13,
    'Hawaii': 15, 'Idaho': 16, 'Illinois': 17, 'Indiana': 18, 'Iowa': 19, 'Kansas': 20,
    'Kentucky': 21, 'Louisiana': 22, 'Maine': 23, 'Maryland': 24, 'Massachusetts': 25,
    'Michigan': 26, 'Minnesota': 27, 'Mississippi': 28, 'Missouri': 29, 'Montana': 30,
    'Nebraska': 31, 'Nevada': 32, 'New Hampshire': 33, 'New Jersey': 34, 'New Mexico': 35,
    'New York': 36, 'North Carolina': 37, 'North Dakota': 38, 'Ohio': 39, 'Oklahoma': 40,
    'Oregon': 41, 'Pennsylvania': 42, 'Rhode Island': 44, 'South Carolina': 45,
    'South Dakota': 46, 'Tennessee': 47, 'Texas': 48, 'Utah': 49, 'Vermont': 50,
    'Virginia': 51, 'Washington': 53, 'West Virginia': 54, 'Wisconsin': 55, 'Wyoming': 56,
}
VALID_FIPS = set(STATE_NAME_TO_FIPS.values())


def as_numeric(series: pd.Series) -> pd.Series:
    """Coerce a series to numeric, replacing unparseable values with NaN."""
    return pd.to_numeric(series, errors='coerce')


def is_empty_cell(value) -> bool:
    """Check if a cell value is effectively empty (NaN, blank, or literal 'nan')."""
    if pd.isna(value):
        return True
    text = str(value).strip()
    return text == '' or text.lower() == 'nan'


def non_empty_tokens(values) -> list[str]:
    """Return a list of non-empty string tokens from a row of values."""
    return [str(v).strip() for v in values if not is_empty_cell(v)]


def find_fsa_sheet_and_header(xls: pd.ExcelFile):
    """Scan all sheets in the FSA workbook to locate the header row containing
    'Location', 'Balance', and 'Borrower' columns."""
    for sheet_name in xls.sheet_names:
        raw = pd.read_excel(FSA_FILE, sheet_name=sheet_name, header=None)
        for i in range(min(150, len(raw))):
            row_text = ' '.join(non_empty_tokens(raw.iloc[i].tolist())).lower()
            if 'location' in row_text and 'balance' in row_text and 'borrower' in row_text:
                return sheet_name, raw, i
    raise RuntimeError('FSA header row not found — check that the uploaded file is the correct FSA portfolio XLS.')


def extract_fsa_reporting_period(raw: pd.DataFrame) -> str:
    """Extract the 'Data as of ...' reporting period string from the FSA file header rows."""
    for i in range(min(50, len(raw))):
        line = ' '.join(non_empty_tokens(raw.iloc[i].tolist()))
        match = re.search(r'Data as of\s+(.+)$', line, flags=re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return 'Unknown'


def find_header_col_index(header_cells, required_bits: list[str]):
    """Find the column index in a header row where all required_bits appear in the cell text."""
    for idx, cell in enumerate(header_cells):
        text = str(cell).strip().lower()
        if all(bit in text for bit in required_bits):
            return idx
    return None


def load_fsa_state_level(fsa_path: Path):
    """Load and parse the FSA portfolio-by-location XLS into a clean state-level DataFrame.
    Returns (DataFrame, reporting_period_str, sheet_name_used)."""
    xls = pd.ExcelFile(fsa_path)
    sheet_name, raw, header_row = find_fsa_sheet_and_header(xls)
    reporting_period = extract_fsa_reporting_period(raw)

    header_cells = raw.iloc[header_row].tolist()
    loc_idx = find_header_col_index(header_cells, ['location'])
    bal_idx = find_header_col_index(header_cells, ['balance', 'billion'])
    bor_idx = find_header_col_index(header_cells, ['borrower', 'thousand'])
    if None in (loc_idx, bal_idx, bor_idx):
        raise RuntimeError('Missing required FSA columns (Location, Balance, Borrowers). File format may have changed.')

    records = []
    for i in range(header_row + 1, len(raw)):
        row = raw.iloc[i].tolist()
        state_val = row[loc_idx] if loc_idx < len(row) else None
        bal_val = row[bal_idx] if bal_idx < len(row) else None
        bor_val = row[bor_idx] if bor_idx < len(row) else None

        if is_empty_cell(state_val) and is_empty_cell(bal_val) and is_empty_cell(bor_val):
            break

        state_name = str(state_val).strip() if not is_empty_cell(state_val) else ''
        if state_name not in STATE_NAME_TO_FIPS:
            continue

        records.append({
            'state': state_name,
            'state_fips': STATE_NAME_TO_FIPS[state_name],
            'balance_billions': bal_val,
            'borrowers_thousands': bor_val,
            'reporting_period': reporting_period,
        })

    fsa = pd.DataFrame(records)
    fsa['balance_billions'] = as_numeric(fsa['balance_billions'])
    fsa['borrowers_thousands'] = as_numeric(fsa['borrowers_thousands'])
    fsa['state_fips'] = fsa['state_fips'].astype(int)
    return fsa, reporting_period, sheet_name


def load_acs_state_level(acs_path: Path):
    """Load ACS median household income CSV. Skips the descriptive sub-header row,
    extracts state FIPS from GEO_ID, and filters to valid 50 states + DC."""
    acs = pd.read_csv(acs_path, skiprows=[1])
    acs.columns = [str(c).strip().replace('\ufeff', '') for c in acs.columns]

    acs = acs[['GEO_ID', 'NAME', 'B19013_001E']].copy()
    acs.columns = ['GEO_ID', 'state_name', 'median_income']
    acs['state_fips'] = acs['GEO_ID'].astype(str).str.extract(r'(\d{2})$')[0]
    acs['state_fips'] = pd.to_numeric(acs['state_fips'], errors='coerce')
    acs['median_income'] = as_numeric(acs['median_income'])

    acs = acs.dropna(subset=['state_fips']).copy()
    acs['state_fips'] = acs['state_fips'].astype(int)
    acs = acs[acs['state_fips'].isin(VALID_FIPS)].copy()
    return acs[['state_name', 'state_fips', 'median_income']]


def load_cfpb_state_level(cfpb_path: Path):
    """Load CFPB consumer credit trends CSV. Extracts state FIPS, abbreviation,
    and year-over-year origination change value."""
    cfpb = pd.read_csv(cfpb_path)
    cfpb.columns = [str(c).strip() for c in cfpb.columns]

    cfpb = cfpb[['fips_code', 'state_abbr', 'value']].copy()
    cfpb.columns = ['state_fips', 'state_abbr', 'yoy_change']
    cfpb['state_fips'] = pd.to_numeric(cfpb['state_fips'], errors='coerce')
    cfpb['yoy_change'] = as_numeric(cfpb['yoy_change'])

    cfpb = cfpb.dropna(subset=['state_fips']).copy()
    cfpb['state_fips'] = cfpb['state_fips'].astype(int)
    cfpb = cfpb[cfpb['state_fips'].isin(VALID_FIPS)].copy()
    return cfpb[['state_fips', 'state_abbr', 'yoy_change']]


# ── Load all three datasets with error handling ──
# Each load is wrapped in try/except so that a single bad file produces a
# clear error message rather than an opaque traceback.

try:
    fsa_raw, fsa_reporting_period, fsa_sheet_used = load_fsa_state_level(FSA_FILE)
    print(f'FSA loaded: {len(fsa_raw)} rows  (reporting period: {fsa_reporting_period})')
except Exception as e:
    raise SystemExit(f'PIPELINE HALTED — FSA load failed: {e}')

try:
    acs_raw = load_acs_state_level(ACS_FILE)
    print(f'ACS loaded: {len(acs_raw)} rows')
except Exception as e:
    raise SystemExit(f'PIPELINE HALTED — ACS load failed: {e}')

try:
    cfpb_raw = load_cfpb_state_level(CFPB_FILE)
    print(f'CFPB loaded: {len(cfpb_raw)} rows')
except Exception as e:
    raise SystemExit(f'PIPELINE HALTED — CFPB load failed: {e}')

print(f'\nAll 3 datasets ingested successfully.')

## Section 3 - Standardize

Convert units and enforce consistent state-level keys (`state_fips` as integer).


In [33]:
fsa_std = fsa_raw.copy()
fsa_std['total_balance'] = fsa_std['balance_billions'] * 1_000_000_000
fsa_std['borrowers'] = fsa_std['borrowers_thousands'] * 1_000
fsa_std['state_fips'] = fsa_std['state_fips'].astype(int)

acs_std = acs_raw.copy()
acs_std['median_income'] = as_numeric(acs_std['median_income'])
acs_std['state_fips'] = acs_std['state_fips'].astype(int)

cfpb_std = cfpb_raw.copy()
cfpb_std['state_fips'] = cfpb_std['state_fips'].astype(int)
cfpb_std['yoy_change'] = as_numeric(cfpb_std['yoy_change'])

print('Standardized dtypes:')
print('FSA state_fips:', fsa_std['state_fips'].dtype)
print('ACS state_fips:', acs_std['state_fips'].dtype)
print('CFPB state_fips:', cfpb_std['state_fips'].dtype)


Standardized dtypes:
FSA state_fips: int64
ACS state_fips: int64
CFPB state_fips: int64


## Section 4 - Join

Merge the three datasets on `state_fips`.


In [34]:
merged = (
    fsa_std[['state', 'state_fips', 'total_balance', 'borrowers', 'reporting_period']]
    .merge(acs_std[['state_name', 'state_fips', 'median_income']], on='state_fips', how='inner')
    .merge(cfpb_std[['state_fips', 'state_abbr', 'yoy_change']], on='state_fips', how='inner')
)
merged['state'] = merged['state_name'].fillna(merged['state'])
merged = merged.drop(columns=['state_name']).sort_values('state').reset_index(drop=True)

print('Joined rows:', len(merged))
merged.head()


Joined rows: 51


,state,state_fips,total_balance,borrowers,reporting_period,median_income,state_abbr,yoy_change
0,Alabama,1,2.400000e+10,634400.0,"Sept. 30, 2025",63999,AL,-0.053603
1,Alaska,2,2.200000e+09,61800.0,"Sept. 30, 2025",92788,AK,1.458374
2,Arizona,4,3.130000e+10,864000.0,"Sept. 30, 2025",79964,AZ,0.138331
3,Arkansas,5,1.310000e+10,384600.0,"Sept. 30, 2025",60773,AR,0.653522
4,California,6,1.468000e+11,3713400.0,"Sept. 30, 2025",99122,CA,0.097834


## Section 5 - Compute Metrics

Create prototype state-level indicators and decision-support labels for BorrowSmart.


In [ ]:
state_metrics = merged.copy()

# Avoid divide-by-zero and propagate missingness safely.
state_metrics['average_debt_per_borrower'] = np.where(
    state_metrics['borrowers'] > 0,
    state_metrics['total_balance'] / state_metrics['borrowers'],
    np.nan,
)
state_metrics['debt_to_income_ratio'] = np.where(
    state_metrics['median_income'] > 0,
    state_metrics['average_debt_per_borrower'] / state_metrics['median_income'],
    np.nan,
)


# ── Risk label thresholds ──
# Thresholds were chosen based on the distribution of computed DTI ratios across
# all 51 states.  A ratio below 0.45 means average student debt is less than 45%
# of a state's median household income (manageable).  Between 0.45 and 0.60 is
# elevated but not extreme (medium).  Above 0.60 flags states where average debt
# exceeds 60% of median income — an outsized burden that may warrant prioritized
# advisory outreach.  These cutpoints roughly correspond to the lower quartile
# boundary (~0.45) and upper quartile boundary (~0.60) of the observed data.
RISK_LOW_THRESHOLD = 0.45
RISK_HIGH_THRESHOLD = 0.60


def classify_risk_label(value):
    if pd.isna(value):
        return np.nan
    if value < RISK_LOW_THRESHOLD:
        return 'Low'
    if value < RISK_HIGH_THRESHOLD:
        return 'Medium'
    return 'High'


state_metrics['risk_label'] = state_metrics['debt_to_income_ratio'].apply(classify_risk_label)

state_metrics[
    [
        'state',
        'state_fips',
        'state_abbr',
        'total_balance',
        'borrowers',
        'median_income',
        'average_debt_per_borrower',
        'debt_to_income_ratio',
        'yoy_change',
        'risk_label',
        'reporting_period',
    ]
].head()

## Section 6 - Validation

Prototype validation checks for row counts, join completeness, missingness, reporting period extraction, and export readiness.


In [36]:
print('Prototype validation checks')
print({
    'FSA_rows': int(len(fsa_std)),
    'ACS_rows': int(len(acs_std)),
    'CFPB_rows': int(len(cfpb_std)),
})

print('\nJoined states before export:', int(len(state_metrics)))
expected_states = 50
actual_states = state_metrics[state_metrics['state'] != 'District of Columbia'].shape[0]
print('All 50 states present before export:', actual_states == expected_states)

critical_indicator_columns = [
    'state',
    'state_fips',
    'total_balance',
    'borrowers',
    'median_income',
    'average_debt_per_borrower',
    'debt_to_income_ratio',
]
critical_missing_counts = state_metrics[critical_indicator_columns].isna().sum()
print('\nMissing values in critical indicator columns:')
print(critical_missing_counts.to_string())

print('\nExtracted FSA reporting period:', fsa_reporting_period)


Prototype validation checks
{'FSA_rows': 51, 'ACS_rows': 51, 'CFPB_rows': 51}

Joined states before export: 51
All 50 states present before export: True

Missing values in critical indicator columns:
state                        0
state_fips                   0
total_balance                0
borrowers                    0
median_income                0
average_debt_per_borrower    0
debt_to_income_ratio         0

Extracted FSA reporting period: Sept. 30, 2025


## Section 7 - Outputs

Create prototype export-ready outputs that could support advisor decision making.


In [37]:
state_metrics = state_metrics.copy()

# Drop rows without the main metric before ranking.
rankable = state_metrics.dropna(subset=['debt_to_income_ratio']).copy()
top10_burden_states = rankable.sort_values('debt_to_income_ratio', ascending=False).head(10).reset_index(drop=True)

# Replace NaN in exported tables with blanks to avoid showing NaN in downstream inspection.
state_metrics_export = state_metrics.fillna('')
top10_export = top10_burden_states.fillna('')

state_metrics_export.to_csv('borrowsmart_prototype_state_metrics.csv', index=False)
top10_export.to_csv('borrowsmart_prototype_top10_burden_states.csv', index=False)

print('Exported prototype files:')
print('  borrowsmart_prototype_state_metrics.csv')
print('  borrowsmart_prototype_top10_burden_states.csv')

print('\nTop 10 burden states:')
top10_burden_states[['state', 'state_abbr', 'debt_to_income_ratio', 'risk_label', 'yoy_change']].round(4).fillna('')


Exported prototype files:
  borrowsmart_prototype_state_metrics.csv
  borrowsmart_prototype_top10_burden_states.csv

Top 10 burden states:


,state,state_abbr,debt_to_income_ratio,risk_label,yoy_change
0,Mississippi,MS,0.6604,High,-0.0905
1,Alabama,AL,0.5911,Medium,-0.0536
2,Louisiana,LA,0.5735,Medium,0.2864
3,Arkansas,AR,0.5605,Medium,0.6535
4,South Carolina,SC,0.5566,Medium,1.2341
5,West Virginia,WV,0.5521,Medium,1.3519
6,Georgia,GA,0.5447,Medium,0.2120
7,New Mexico,NM,0.5414,Medium,-0.0077
8,Tennessee,TN,0.5394,Medium,0.1853
9,North Carolina,NC,0.5384,Medium,0.4508


In [38]:
pipeline_end = time.time()

critical_export_columns = [
    'state',
    'state_fips',
    'state_abbr',
    'average_debt_per_borrower',
    'median_income',
    'debt_to_income_ratio',
    'risk_label',
]
critical_export_missing_counts = state_metrics[critical_export_columns].isna().sum()
top10_has_10_rows = 'top10_burden_states' in globals() and len(top10_burden_states) == 10

print('Validation summary')
print({
    'FSA_rows': int(len(fsa_std)),
    'ACS_rows': int(len(acs_std)),
    'CFPB_rows': int(len(cfpb_std)),
    'joined_states': int(len(state_metrics)),
    'all_50_states_present': state_metrics[state_metrics['state'] != 'District of Columbia'].shape[0] == 50,    'critical_missing_values_total': int(critical_export_missing_counts.sum()),
    'top10_burden_states_has_10_rows': top10_has_10_rows,
})

print('\nMissing values in critical exported fields:')
print(critical_export_missing_counts.to_string())

print(f'\nTotal pipeline runtime: {pipeline_end - pipeline_start:.2f} seconds')


Validation summary
{'FSA_rows': 51, 'ACS_rows': 51, 'CFPB_rows': 51, 'joined_states': 51, 'all_50_states_present': True, 'critical_missing_values_total': 0, 'top10_burden_states_has_10_rows': True}

Missing values in critical exported fields:
state                        0
state_fips                   0
state_abbr                   0
average_debt_per_borrower    0
median_income                0
debt_to_income_ratio         0
risk_label                   0

Total pipeline runtime: 0.19 seconds


## Section 8 - Visualization

Bar chart of the top 10 states by prototype debt-to-income burden ratio.


In [ ]:
plot_df = top10_burden_states.copy().sort_values('debt_to_income_ratio', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

# Color bars by risk tier for visual emphasis
colors = ['#d32f2f' if r == 'High' else '#f57c00' if r == 'Medium' else '#1f77b4'
          for r in plot_df['risk_label']]

ax.barh(plot_df['state_abbr'], plot_df['debt_to_income_ratio'], color=colors)
ax.set_xlabel('Debt-to-Income Ratio (Average Debt per Borrower / Median Household Income)')
ax.set_ylabel('State')
ax.set_title('Top 10 States by BorrowSmart Prototype Debt Burden Indicator')
ax.grid(axis='x', linestyle='--', alpha=0.3)

# Add value labels on each bar
for i, (val, state) in enumerate(zip(plot_df['debt_to_income_ratio'], plot_df['state_abbr'])):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()

# Save chart to file (in addition to inline display)
CHART_PATH = 'borrowsmart_top10_burden_chart.png'
fig.savefig(CHART_PATH, dpi=150, bbox_inches='tight')
print(f'Chart saved to {CHART_PATH}')

plt.show()

## Section 9 - Documentation

### What the debt burden indicator represents
The BorrowSmart prototype debt burden indicator (`debt_to_income_ratio`) is a prototype metric that estimates how large the average student loan debt per borrower is relative to a state's median household income. Higher values may reflect possible state-level repayment strain relative to typical household earnings in that state.

This metric is intended as contextual decision support, not as an automated recommendation.

Because the prototype uses aggregated state-level data, the metric should be interpreted as a regional context indicator rather than a measure of individual borrower risk.

### Limitations of state averages
- State averages can hide within-state variation across income groups, age groups, and institutions.
- Median household income is a household-level measure, not borrower-only income.
- FSA balances and borrower counts are aggregate portfolio figures and do not capture repayment plan status or delinquency risk directly.
- CFPB YoY origination change is a separate market trend signal and should not be interpreted as borrower-level distress.

### Data sources used
- U.S. Department of Education Federal Student Aid (Direct Loan Portfolio by Borrower Location)
- U.S. Census Bureau ACS 5-year estimates (`B19013` median household income)
- CFPB student loan map data (`map_data_STU.csv`, YoY change in student loan originations)
